In [5]:
# !pip install unsloth

In [2]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
max_seq_length = 2048 
dtype = None          
load_in_4bit = True   

In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/kaggle/input/models/hridaypatel75/cardiology-domain-specific-fine-tuning/transformers/default/1/kaggle/working/qwen2.5-3b-cardiology-lora", 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.6.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [7]:
for name, param in model.named_parameters():
    if "lora" in name:
        param.requires_grad = True

In [8]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", 
)

Unsloth: Will map <|im_end|> to EOS = <|endoftext|>.


In [9]:
sft_dataset = load_dataset("ruslanmv/ai-medical-chatbot", split="train")

README.md:   0%|          | 0.00/863 [00:00<?, ?B/s]

dialogues.parquet:   0%|          | 0.00/142M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/256916 [00:00<?, ? examples/s]

In [10]:
def format_chat_interaction(example):
    patient_text = str(example.get("Patient", ""))
    doctor_text = str(example.get("Doctor", ""))

    chat = [
        {"role": "system", "content": "You are a helpful, empathetic, and expert medical professional. Explain medical concepts clearly to the patient."},
        {"role": "user", "content": patient_text},
        {"role": "assistant", "content": doctor_text}
    ]
    
    formatted_text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=False)
    
    return {"text": formatted_text}

In [11]:
original_columns = sft_dataset.column_names
sft_dataset = sft_dataset.map(format_chat_interaction, remove_columns=original_columns)

Map:   0%|          | 0/256916 [00:00<?, ? examples/s]

In [12]:
args = SFTConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-5, 
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs_phase2",
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    dataset_num_proc = 1, 
)

In [15]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = sft_dataset,
    args = args,
)

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/256916 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free enabled, enabling faster training.


In [16]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 256,916 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.043886
2,2.836267
3,3.035457
4,3.207531
5,2.950854
6,2.871973
7,3.124665
8,2.909719
9,2.900943
10,2.736751


Unsloth: Restored added_tokens_decoder metadata in outputs_phase2/checkpoint-60/tokenizer_config.json.


In [17]:
final_adapter_name = "qwen2.5-3b-cardio-chat"
model.save_pretrained(final_adapter_name)
tokenizer.save_pretrained(final_adapter_name)

Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-cardio-chat/tokenizer_config.json.


('qwen2.5-3b-cardio-chat/tokenizer_config.json',
 'qwen2.5-3b-cardio-chat/chat_template.jinja',
 'qwen2.5-3b-cardio-chat/tokenizer.json')

In [27]:
!zip -r qwen25-3b-cardio-chat /kaggle/working/qwen2.5-3b-cardio-chat

  adding: kaggle/working/qwen2.5-3b-cardio-chat/ (stored 0%)
  adding: kaggle/working/qwen2.5-3b-cardio-chat/README.md (deflated 65%)
  adding: kaggle/working/qwen2.5-3b-cardio-chat/tokenizer_config.json (deflated 90%)
  adding: kaggle/working/qwen2.5-3b-cardio-chat/chat_template.jinja (deflated 59%)
  adding: kaggle/working/qwen2.5-3b-cardio-chat/adapter_model.safetensors (deflated 8%)
  adding: kaggle/working/qwen2.5-3b-cardio-chat/adapter_config.json (deflated 57%)
  adding: kaggle/working/qwen2.5-3b-cardio-chat/tokenizer.json (deflated 81%)


In [19]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048, padding_idx=151665)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [22]:
messages = [
    {"role": "system", "content": "You are a helpful, empathetic, and expert medical professional. Explain medical concepts clearly to the patient."},
    {"role": "user", "content": "Hi doctor, my smartwatch recently alerted me that my resting heart rate is over 110 bpm, and I've been feeling a weird fluttering sensation in my chest. What could this be, and what should I do?"}
]

In [23]:
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

In [24]:
with model.disable_adapter():
    base_outputs = model.generate(
        input_ids=inputs, 
        max_new_tokens=200, 
        use_cache=True,
        temperature=0.7,          
        top_p=0.9,                
        repetition_penalty=1.2,   
        pad_token_id=tokenizer.eos_token_id, 
        eos_token_id=tokenizer.eos_token_id
    )
base_response = tokenizer.batch_decode(base_outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]
print(base_response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

Resting Heart Rate (RHR) above 11k beats per minute can indicate an elevated level of stress or anxiety which may cause your RHR to increase temporarily. The fluttering sensation you're experiencing might also suggest palpitations due to increased adrenaline levels.

It's important not to panic yourself with any medication without consulting a healthcare provider first as it will depend on various factors such as age, overall health condition etc., but here’s some general advice:

- Try reducing physical activity for few days.
- Avoid caffeine intake if possible because it increases blood pressure leading to higher HRV.
- Practice deep breathing exercises regularly; they help lower cortisol levels associated with high-stress situations.
- If symptoms persist after trying these methods out, seek immediate attention from a physician who can perform tests like ECG/EKG monitoring during rest periods along with other diagnostic procedures depending upon their findings.

Remember always cons

In [25]:
chat_outputs = model.generate(
    input_ids=inputs, 
    max_new_tokens=200, 
    use_cache=True,
    temperature=0.7,        
    top_p=0.9,                
    repetition_penalty=1.2,   
    pad_token_id=tokenizer.eos_token_id, 
    eos_token_id=tokenizer.eos_token_id
)

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [26]:
response = tokenizer.batch_decode(chat_outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]
print(response)

Hello! It's important for you to consult with your healthcare provider as soon as possible regarding these symptoms. However, based on some information provided by yourself, it seems like there might be an issue related to your cardiovascular system or possibly other factors such as stress levels affecting your body.

Resting Heart Rate (RHR) above 110 beats per minute can indicate several conditions including but not limited to:

- **Cardiac Arrhythmias**: Irregular heartbeat patterns which may cause palpitations.
  
- **Anxiety/Stress**: Sometimes anxiety triggers rapid breathing leading to increased RHR along with sensations of discomfort or uneasiness known medically as "palpitations".

- Other less common causes include thyroid disorders, dehydration etc., however without proper examination one cannot make definitive conclusions about underlying pathology just from rest HR alone unless accompanied by additional clinical signs & symptoms specific enough to suggest certain diagnoses